# Type a data structure

Python has four common shapes for "a record of named fields": a plain `dict`, a `TypedDict`, a `NamedTuple`, and a `@dataclass`. They overlap — and the right choice depends on what the data is and how it'll be used. This recipe is the decision tree.

## The four options at a glance

| Shape | Syntax for access | Mutability | Best for |
| --- | --- | --- | --- |
| `dict` | `d["key"]` | Mutable | Unstructured / dynamic keys |
| `TypedDict` | `d["key"]` | Mutable | Dict-shaped data from APIs or config |
| `NamedTuple` | `n.key` | Immutable | Small, immutable records |
| `@dataclass` | `d.key` | Configurable | General-purpose records, esp. with methods |


## Plain `dict[K, V]`

Reach for this when the keys are dynamic or the dict is a genuine mapping of key-type to value-type, not a record:

In [ ]:
# Counters, lookups, caches — the keys are data, not named fields
char_counts: dict[str, int] = {}
for ch in "hello":
    char_counts[ch] = char_counts.get(ch, 0) + 1
print(char_counts)

If you find yourself writing `dict[str, str | int | bool | list[...]]`, you've outgrown plain dicts — use a `TypedDict` or a dataclass.

## `TypedDict` — structured dicts

When the data is *already* a dict (a JSON payload, a parsed YAML file, a pandas record) and you want to document the shape without converting:

In [ ]:
from typing import TypedDict
try:
    from typing import NotRequired           # Python 3.11+
except ImportError:
    from typing_extensions import NotRequired

class UserRecord(TypedDict):
    id: int
    name: str
    active: bool
    email: NotRequired[str]           # this key may be missing

# Use it just like a dict
alice: UserRecord = {"id": 1, "name": "Alice", "active": True}
print(alice["name"])

**When to use**: data that naturally lives as a dict (parsing, serialisation, interop with external systems) and you want editor autocomplete and `mypy` checks on the keys.

**When not to use**: if the data is built up in Python and read/written many times, a dataclass gives you better ergonomics (`user.name` vs `user["name"]`, and IDE features like rename-symbol work on attributes but not dict keys).

## `NamedTuple` — small immutable records

Good for tiny records where you want attribute access *and* tuple unpacking, and you don't need methods:

In [ ]:
from typing import NamedTuple

class Point(NamedTuple):
    x: float
    y: float

p = Point(3.0, 4.0)
print(p.x, p.y)         # attribute access
x, y = p                # tuple unpacking
print(x + y)
print(p == (3.0, 4.0))  # equal to regular tuples

**When to use**: small, hashable, equality-on-values records. Coordinates, RGB values, results-of-a-split. The immutability is a feature.

**When not to use**: if you want methods beyond trivial ones, or if mutation matters. The `tuple` inheritance leaks: `Point(3, 4)[0] == 3` is true, which is occasionally confusing.

## `@dataclass` — the default choice

For most records you build and manipulate in Python, a dataclass is the right tool:

In [ ]:
from dataclasses import dataclass

@dataclass
class User:
    id: int
    name: str
    active: bool = True

    def display(self) -> str:
        return f"{self.name} (id={self.id})"

alice = User(id=1, name="Alice")
print(alice.display())
print(alice)       # auto-generated __repr__

**When to use**: the default for records with named fields. Auto-generated `__init__`, `__repr__`, `__eq__`. You can add methods. It supports `frozen=True` for immutability, `slots=True` for memory efficiency, inheritance, and more. See the [dataclass parameters reference](../../classes-and-objects/reference/dataclass-parameters.md).

**When not to use**: when the data comes from outside Python as a dict — use `TypedDict`. When you need tuple behaviour — use `NamedTuple`. When the record is so small that a class would be ceremony — use a 2-tuple or a dict literal.

## The decision tree

1. **Is it a keyed lookup with dynamic keys?** (counters, caches, indexes) → `dict[K, V]`.
2. **Is the data already a dict from an external source?** → `TypedDict`.
3. **Is it a tiny immutable record you want to unpack?** → `NamedTuple`.
4. **Anything else?** → `@dataclass`.

This covers 95% of cases. The remaining 5% are usually "I need a class with real behaviour" — reach for a regular class with an `__init__` and methods.

## Typing a nested structure

Records often contain records. Type-annotating the nesting is the same as any other:

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Address:
    street: str
    postcode: str

@dataclass
class Customer:
    name: str
    age: int
    address: Address                            # nested dataclass
    tags: list[str] = field(default_factory=list)
    metadata: dict[str, str] = field(default_factory=dict)

c = Customer(
    name="Alice",
    age=30,
    address=Address("42 High St", "SW1A 1AA"),
)
print(c)

`field(default_factory=list)` is the idiom for mutable defaults in dataclasses — it runs `list()` each time a new `Customer` is constructed, so the default isn't shared between instances. See the [`@dataclass` parameters reference](../../classes-and-objects/reference/dataclass-parameters.md) for the full set of options.

## Typing collections of records

A list of users, a dict of customers-by-id — nothing special, just compose:

In [ ]:
users: list[User] = [
    User(id=1, name="Alice"),
    User(id=2, name="Bob", active=False),
]

by_id: dict[int, User] = {u.id: u for u in users}

print(users)
print(by_id[1])

## Tips

- **Default to dataclass.** It's expressive, well-supported, and scales from tiny records to complex ones.
- **Use `TypedDict` for wire formats.** If data arrives as a dict and leaves as a dict, don't convert it for the sake of typing.
- **Don't convert between shapes casually.** Each conversion is code you have to maintain and a place where types can diverge from runtime.
- **Keep the nested depth reasonable.** Past two levels, consider splitting into named types — a giant nested annotation is a sign you've found a record that deserves its own class.